# Cache-first lead-time modes-of-variability teleconnection metrics

This notebook diagnoses lead-dependent teleconnections between one upstream mode-of-variability (MOV) index and one or more downstream gridded variables. It is designed around the products created by:

- `4_refactor_mov_analysis.ipynb`: compact forecast/reference MOV indices and valid times;
- `1a_refactor_atm_leadtime_acc_skill_map.ipynb`: atmospheric prepared anomaly bundles and prepared observations;
- `1b_refactor_lnd_leadtime_acc_skill_map.ipynb`: prepared land forecast and reference bundles.

**Cache-only rule:** this notebook reads analysis-ready products. It does not open raw archives, derive EOFs or station indices, regrid, or reproduce 1a/1b/4 preprocessing. Missing or ambiguous products stop with an actionable error.

For each system, initialization month, and lead, the notebook estimates:

1. the observed teleconnection map: correlation of the observed MOV index with the observed downstream anomaly;
2. the forecast teleconnection map: correlation of the ensemble-mean forecast MOV index with the ensemble-mean downstream anomaly across initialization years;
3. scalar fidelity metrics comparing forecast and observed maps: spatial pattern correlation, centered spatial RMSE, regression slope/amplitude ratio, sign agreement, significant-area fractions, and sample counts.

Correlation maps, p-values, and scalar metrics are saved to a provenance-rich NetCDF file. A lead-summary figure and selected map panels are also saved.

In [ ]:
import os
import sys
from pathlib import Path

_repo_override = os.environ.get("ESP_LAB_REPO_ROOT")
_repo_candidates = (
    [Path(_repo_override).expanduser().resolve()]
    if _repo_override
    else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
)
REPO_ROOT = next((p for p in _repo_candidates if (p / "workflows" / "diagnostics").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside ESP-Lab or set ESP_LAB_REPO_ROOT to its checkout.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAVE_CARTOPY = True
except ImportError:
    HAVE_CARTOPY = False

try:
    from scipy import stats as scipy_stats
except ImportError as exc:
    raise ImportError("This notebook requires scipy for p-values") from exc

from workflows.diagnostics import mov_teleconnections as mov_telecon

%matplotlib inline


## 1. User configuration

Normally only this cell needs editing. `upstream_mode` must be one of the modes produced by the MOV workflow. `downstream_variables` may mix atmospheric and land quantities. The selected seasonal cache contract matches the centered three-month means used in the source notebooks.

In [ ]:
CONFIG = {
    "paths": {
        "diag_root": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag",
        "output_dir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/teleconnections",
        "figure_dir": "/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/teleconnections",
    },
    "selection": {
        "upstream_mode": "NAM",
        "downstream_variables": ["TREFHT", "PRECT", "H2OSNO"],
        "systems": ["E3SM-FOSIRL", "E3SM-Reanalysis", "E3SM-4DEnVarOcn"],
        "init_months": [5, 11],
        "leads": None,  # None = all common complete seasonal leads
        "verification_years": (1981, 2011),
        "climatology_years": (1981, 2010),
        "target_grid": "latlon_1.0x1.0_periodic-True",
    },
    "analysis": {
        "detrend": True,
        "alpha": 0.10,
        "minimum_years": 20,
        "latitude_bounds": (-80.0, 80.0),
        "area_weighted": True,
        "sign_agreement_threshold": 0.0,
    },
    "cache": {
        "mode": "auto",  # auto | rebuild | require
        "allow_ambiguous_matches": False,
    },
    "figures": {
        "map_leads": [3, 9, 15, 21],
        "dpi": 200,
        "cmap": "RdBu_r",
        "correlation_limits": (-1.0, 1.0),
    },
}

E3SM_CASES = mov_telecon.E3SM_CASES
MOV_MODES = mov_telecon.MOV_MODES
DOWNSTREAM_VARIABLES = mov_telecon.DOWNSTREAM_VARIABLES

index_name = CONFIG["selection"]["upstream_mode"].upper().strip()
if index_name not in MOV_MODES:
    raise ValueError(f"Unknown MOV mode {index_name!r}; choose from {list(MOV_MODES)}")
unknown = set(CONFIG["selection"]["downstream_variables"]) - set(DOWNSTREAM_VARIABLES)
if unknown:
    raise ValueError(f"Unconfigured downstream variables: {sorted(unknown)}")
if CONFIG["cache"]["mode"] not in {"auto", "rebuild", "require"}:
    raise ValueError("cache.mode must be auto, rebuild, or require")


## 2. Cache registry and discovery

This registry separates scientific names from physical files. MOV paths are resolved from the shared `modes_manifest.json` first, matching the contract used in notebook 4, and then from the deterministic legacy layout. Forecast products must contain either the skill-ready pair `mode_index_skill`/`valid_time_skill` or the base pair `mode_index`/`valid_time`; references must contain `mode_index`.

Atmospheric prepared bundles contain `anomaly`, `climatology`, and `time`; land bundles contain the configured field plus `time`. Observation caches contain `observation` (1a) or the land reference variable (1b). Hashed/versioned downstream products are selected by metadata, not merely by filename.

In [ ]:
DIAG_ROOT = Path(CONFIG["paths"]["diag_root"])
OUTPUT_DIR = Path(CONFIG["paths"]["output_dir"])
FIGURE_DIR = Path(CONFIG["paths"]["figure_dir"])

# Expose reusable workflow helpers
select_cache = mov_telecon.select_cache
upstream_paths = mov_telecon.upstream_mov_paths
resolve_downstream_paths = mov_telecon.resolve_downstream_paths
load_modes_manifest = mov_telecon.load_modes_manifest


In [ ]:
# Inventory only: verify availability of all required upstream products
inventory = mov_telecon.build_mov_teleconnection_inventory(CONFIG)
display(inventory)

missing = inventory.query("status == 'missing'")
if not missing.empty:
    raise FileNotFoundError(
        "Required upstream products are missing. Run the appropriate 1a, 1b, or 4a "
        "preparation workflow, then rerun this notebook.\n" + missing.to_string(index=False)
    )
skipped = inventory.query("status == 'skipped'")
if not skipped.empty:
    print(f"Note: {len(skipped)} combination(s) skipped (e.g. system has no land component or mode not computed).")


## 3. Alignment and anomaly helpers

Alignment is by **target year and month from each cache's valid-time coordinate**, never by positional index. This handles possible differences between MOV and downstream lead labels while ensuring that the predictor and response describe the same target season.

The MOV index orientation and standardization are inherited from notebook 4's validated PCMDI EOF products. The 1a `anomaly` field is already lead-dependent drift corrected. Land caches are normalized here only when they are not explicitly marked as anomalies. Observed fields are converted to monthly-climatology anomalies over the configured climatology window, then sampled at the MOV target dates.

In [ ]:
time_year_month = mov_telecon.time_year_month
parse_init_years = mov_telecon.parse_init_years
lead_signature = mov_telecon.lead_signature
match_leads = mov_telecon.match_leads
linear_detrend = mov_telecon.linear_detrend
monthly_anomaly = mov_telecon.monthly_anomaly
lead_anomaly = mov_telecon.lead_anomaly
observed_at_valid_time = mov_telecon.observed_at_valid_time
corr_and_p = mov_telecon.corr_and_p
weighted_spatial_metrics = mov_telecon.weighted_spatial_metrics


## 4. Teleconnection computation

The scalar comparison domain is the intersection of finite forecast/observed maps and the configured latitude band. Cosine-latitude weights are used by default. The forecast map is based on the ensemble mean for both the MOV index and downstream field; this measures the predictable, forced teleconnection rather than within-ensemble weather covariance.

A future extension can add member-wise distributions or paired bootstrap uncertainty without changing the saved core dimensions.

In [ ]:
def compute_one(system, init_month, variable):
    """Compute lead-dependent teleconnection metrics for one (system, init_month, variable) case."""
    return mov_telecon.compute_system_mov_teleconnection(system, init_month, variable, CONFIG)


## 5. Cache metrics

The output key includes all scientific choices. `auto` reuses an exact compatible metrics file; `rebuild` replaces it; `require` refuses to compute if it is absent. Source paths, modification times, and sizes enter the fingerprint so upstream cache changes invalidate downstream teleconnection metrics.

In [ ]:
metrics_ds, out_file, cache_status = mov_telecon.ensure_mov_teleconnection_dataset(CONFIG, inventory=inventory)
fingerprint = metrics_ds.attrs.get("fingerprint", "latest")
print(f"Teleconnection metrics ({cache_status}): {out_file}")
display(metrics_ds)


## 6. Analysis figures

The summary figure shows pattern fidelity and spatial error by lead. Map panels show observed, forecast, and forecast-minus-observed teleconnections for selected leads. Stippling marks grid cells significant at the configured pointwise level; interpret it as descriptive unless a field-significance/FDR extension is added.

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
summary_metrics = ["pattern_correlation", "centered_rmse", "amplitude_ratio", "sign_agreement_fraction"]
ylabels = ["Pattern correlation", "Centered RMSE", "Amplitude ratio", "Sign agreement"]

for variable in metrics_ds.variable.values:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    for ax, metric, ylabel in zip(axes.ravel(), summary_metrics, ylabels):
        for system in metrics_ds.system.values:
            if str(system) not in E3SM_CASES:
                continue
            for init_month, linestyle in zip(metrics_ds.init_month.values, ("-", "--", ":")):
                series = metrics_ds[metric].sel(variable=variable, system=system, init_month=init_month)
                if series.isnull().all():
                    continue
                ax.plot(
                    series.L, series, marker="o", linestyle=linestyle,
                    color=E3SM_CASES[str(system)]["color"],
                    label=f"{E3SM_CASES[str(system)]['display_name']} init {int(init_month):02d}",
                )
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
    axes[-1, 0].set_xlabel("Seasonal lead coordinate L")
    axes[-1, 1].set_xlabel("Seasonal lead coordinate L")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8)
    fig.suptitle(f"{index_name} teleconnection fidelity: {variable}")
    fig.tight_layout(rect=(0, 0.10, 1, 0.95))
    path = FIGURE_DIR / f"teleconnection_{index_name.replace('.', '')}_{variable}_summary_{fingerprint}.png"
    fig.savefig(path, dpi=CONFIG["figures"]["dpi"], bbox_inches="tight")
    plt.show()
    print("Saved:", path)


In [ ]:
def draw_map(ax, da, title, *, difference=False):
    lat = "lat" if "lat" in da.coords else "latitude"
    lon = "lon" if "lon" in da.coords else "longitude"
    limit = 0.5 if difference else CONFIG["figures"]["correlation_limits"][1]
    cmap = "RdBu_r" if difference else CONFIG["figures"]["cmap"]
    kwargs = dict(
        x=lon, y=lat, ax=ax, cmap=cmap, vmin=-limit, vmax=limit,
        add_colorbar=False, transform=ccrs.PlateCarree() if HAVE_CARTOPY else None
    )
    image = da.plot(**{k: v for k, v in kwargs.items() if v is not None})
    if HAVE_CARTOPY:
        ax.coastlines(linewidth=0.6)
        ax.add_feature(cfeature.BORDERS, linewidth=0.2)
    ax.set_title(title, fontsize=9)
    return image

map_leads = [lead for lead in CONFIG["figures"]["map_leads"] if lead in metrics_ds.L]
for variable in metrics_ds.variable.values:
    for system in metrics_ds.system.values:
        if str(system) not in E3SM_CASES:
            continue
        for init_month in metrics_ds.init_month.values:
            for lead in map_leads:
                subset = metrics_ds.sel(variable=variable, system=system, init_month=init_month, L=lead)
                if subset.observed_correlation.isnull().all() and subset.model_correlation.isnull().all():
                    continue
                projection = ccrs.PlateCarree() if HAVE_CARTOPY else None
                subplot_kw = {"projection": projection} if projection else {}
                fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), subplot_kw=subplot_kw)
                im = draw_map(axes[0], subset.observed_correlation, "Observed")
                draw_map(axes[1], subset.model_correlation, "Forecast")
                diff = subset.model_correlation - subset.observed_correlation
                im_diff = draw_map(axes[2], diff, "Forecast - observed", difference=True)
                fig.colorbar(im, ax=axes[:2], orientation="horizontal", fraction=0.07, pad=0.10, label="Correlation")
                fig.colorbar(im_diff, ax=axes[2], orientation="horizontal", fraction=0.07, pad=0.10, label="Correlation difference")
                fig.suptitle(f"{index_name} -> {variable} | {E3SM_CASES[str(system)]['display_name']} | init {int(init_month):02d}, L={int(lead)}")
                path = FIGURE_DIR / f"teleconnection_{index_name.replace('.', '')}_{variable}_{E3SM_CASES[str(system)]['cache_tag']}_i{int(init_month):02d}_L{int(lead):02d}_{fingerprint}.png"
                fig.savefig(path, dpi=CONFIG["figures"]["dpi"], bbox_inches="tight")
                plt.show()
                print("Saved:", path)


## 7. Interpretation and recommended extensions

- `pattern_correlation` asks whether the forecast reproduces the geographical shape of the observed teleconnection.
- `centered_rmse` measures spatial pattern error after removing each map's area-weighted mean.
- `amplitude_ratio` and `regression_slope` distinguish weak/strong patterns and sign reversal.
- `sign_agreement_fraction` is an intuitive spatial consistency measure.
- Pointwise p-values are saved, but multiple-testing control and field significance are not claimed.

Recommended phase-2 additions are paired bootstrap confidence intervals over years, member-wise teleconnection distributions, partial correlations controlling for an SST index or another MOV mode, lagged index/response seasons, and FDR or field-significance testing. Those should be new configuration options rather than changes to the cache-first input contract.